# 11. SOLUTIONS — Fashion MNIST Classification
---
Complete solutions for all 10 exercises.

## Setup

In [ ]:
try:
    import torch
    print(f"PyTorch version: {torch.__version__}")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "torch", "torchvision", "--break-system-packages", "-q"])
    import torch

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))
])

train_data = datasets.FashionMNIST('./data', train=True,  download=True, transform=transform)
test_data  = datasets.FashionMNIST('./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=64, shuffle=False)

print(f"Training samples : {len(train_data)}")
print(f"Test samples     : {len(test_data)}")
print(f"Image shape      : {train_data[0][0].shape}")

---
## Solution 1 — Visualise the Dataset

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img, label = train_data[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(CLASS_NAMES[label], fontsize=8)
    ax.axis('off')
plt.suptitle("Fashion-MNIST — First 16 Training Samples", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Solution 2 — Build the Network

In [ ]:
class FashionClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),

            nn.Linear(784, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.network(x)

model = FashionClassifier()
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

---
## Solution 3 — Loss Function & Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Loss:      CrossEntropyLoss")
print("Optimizer: Adam (lr=0.001)")

# Why CrossEntropyLoss?
# CrossEntropyLoss is designed for multi-class classification. It applies
# softmax internally to convert raw logits into probabilities, then computes
# the negative log likelihood of the correct class. MSELoss treats classes
# as numbers on a scale (as if class 9 is 'more' than class 0), which makes
# no sense for categories like clothing types. CrossEntropyLoss treats each
# class independently and penalises confident wrong predictions much harder.

---
## Solution 4 — Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()  # enables dropout + batchnorm training behaviour
    total_loss, correct = 0, 0
    for X_batch, y_batch in loader:
        optimizer.zero_grad()                      # clear old gradients
        logits = model(X_batch)                    # forward pass
        loss = criterion(logits, y_batch)          # compute loss
        loss.backward()                            # backprop
        optimizer.step()                           # update weights
        total_loss += loss.item()
        correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)


def evaluate(model, loader, criterion):
    model.eval()   # disables dropout + switches batchnorm to eval mode
    total_loss, correct = 0, 0
    with torch.no_grad():   # no gradient tracking needed during inference
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item()
            correct += (logits.argmax(1) == y_batch).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

---
## Solution 5 — Run the Training Loop

In [ ]:
N_EPOCHS = 10
train_losses, test_losses = [], []
train_accs,   test_accs   = [], []

print(f"{'Epoch':>6} {'Train Loss':>12} {'Train Acc':>11} {'Test Loss':>11} {'Test Acc':>10}")
print("-" * 55)

for epoch in range(1, N_EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    te_loss, te_acc = evaluate(model, test_loader, criterion)

    train_losses.append(tr_loss); test_losses.append(te_loss)
    train_accs.append(tr_acc);   test_accs.append(te_acc)

    print(f"{epoch:>6} {tr_loss:>12.4f} {tr_acc:>10.2%} {te_loss:>11.4f} {te_acc:>10.2%}")

---
## Solution 6 — Plot the Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, N_EPOCHS+1), train_losses, 'b-o', label='Train loss')
axes[0].plot(range(1, N_EPOCHS+1), test_losses,  'r-o', label='Test loss')
axes[0].set_title("Loss Curve")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, N_EPOCHS+1), [a*100 for a in train_accs], 'b-o', label='Train acc')
axes[1].plot(range(1, N_EPOCHS+1), [a*100 for a in test_accs],  'r-o', label='Test acc')
axes[1].set_title("Accuracy Curve")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Fashion-MNIST Training Curves", fontweight='bold')
plt.tight_layout()
plt.show()

# What do the curves tell you about overfitting?
# If train accuracy is significantly higher than test accuracy by epoch 10,
# the model is overfitting — memorising training data rather than generalising.
# With Dropout and BatchNorm, the gap should be small (~2-3%), which means
# the regularisation is doing its job. Fashion-MNIST is harder than digit MNIST
# so some gap is expected — shirts, pullovers and coats look similar to the model.

---
## Solution 7 — Visualise Predictions

In [ ]:
model.eval()
X_sample, y_sample = next(iter(test_loader))

with torch.no_grad():
    logits = model(X_sample[:16])
    probs  = torch.softmax(logits, dim=1)
    preds  = logits.argmax(1)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = X_sample[i].squeeze().numpy()
    ax.imshow(img, cmap='gray')
    pred  = preds[i].item()
    true  = y_sample[i].item()
    conf  = probs[i, pred].item()
    color = 'green' if pred == true else 'red'
    ax.set_title(f"{CLASS_NAMES[pred]}\n({conf:.0%})", color=color, fontsize=7)
    ax.axis('off')

plt.suptitle("Predictions (green=correct, red=wrong)", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Solution 8 — Diagnose Overfitting

In [ ]:
class FashionClassifierNoDrop(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            # No Dropout
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            # No Dropout
            nn.Linear(128, 10),
        )
    def forward(self, x):
        return self.network(x)

model_nodrop = FashionClassifierNoDrop()
opt_nodrop   = optim.Adam(model_nodrop.parameters(), lr=0.001)

nodrop_train_accs, nodrop_test_accs = [], []
for epoch in range(1, N_EPOCHS + 1):
    _, tr_acc = train_epoch(model_nodrop, train_loader, criterion, opt_nodrop)
    _, te_acc = evaluate(model_nodrop, test_loader, criterion)
    nodrop_train_accs.append(tr_acc)
    nodrop_test_accs.append(te_acc)
    print(f"Epoch {epoch}: train={tr_acc:.2%}  test={te_acc:.2%}")

# Plot comparison
plt.figure(figsize=(9, 4))
epochs = range(1, N_EPOCHS + 1)
plt.plot(epochs, [a*100 for a in train_accs],         'b-o',  label='With Dropout — Train')
plt.plot(epochs, [a*100 for a in test_accs],           'b--o', label='With Dropout — Test')
plt.plot(epochs, [a*100 for a in nodrop_train_accs],  'r-o',  label='No Dropout — Train')
plt.plot(epochs, [a*100 for a in nodrop_test_accs],   'r--o', label='No Dropout — Test')
plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)')
plt.title('Dropout vs No Dropout — Overfitting Comparison')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# What difference do you see?
# Without Dropout, train accuracy climbs higher and faster but the gap between
# train and test accuracy widens — the model is memorising the training set.
# With Dropout, train accuracy is slightly lower (Dropout randomly disables
# neurons, making training harder) but test accuracy is similar or better
# because the network is forced to learn more robust, generalisable features
# rather than relying on specific neurons being there.

---
## Solution 9 — Compare Optimizers

In [ ]:
optimizer_configs = [
    ('SGD',           lambda p: optim.SGD(p, lr=0.01)),
    ('SGD+Momentum',  lambda p: optim.SGD(p, lr=0.01, momentum=0.9)),
    ('Adam',          lambda p: optim.Adam(p, lr=0.001)),
]
colors = ['#E74C3C', '#3498DB', '#2ECC71']

plt.figure(figsize=(9, 4))

for (name, opt_fn), col in zip(optimizer_configs, colors):
    m   = FashionClassifier()
    opt = opt_fn(m.parameters())
    accs = []
    for epoch in range(1, N_EPOCHS + 1):
        train_epoch(m, train_loader, criterion, opt)
        _, te_acc = evaluate(m, test_loader, criterion)
        accs.append(te_acc * 100)
    plt.plot(range(1, N_EPOCHS + 1), accs, '-o', color=col, label=name, lw=2)
    print(f"{name:15s}  final test acc: {accs[-1]:.2f}%")

plt.xlabel('Epoch'); plt.ylabel('Test Accuracy (%)')
plt.title('Optimizer Comparison on Fashion-MNIST')
plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Which converges fastest? Which is most stable?
# Adam converges fastest and most smoothly — it adapts the learning rate
# per weight, so it handles the varied gradient magnitudes across layers well.
# SGD+Momentum is close behind: the momentum term helps it build speed in
# the right direction and smooth out noisy gradients.
# Plain SGD is slowest and most erratic — it takes equally-sized steps
# regardless of the gradient history, so it struggles to navigate the
# loss surface efficiently.
# Yes, this matches Notebook 8 exactly.

---
## Solution 10 — Find Confident Mistakes

In [ ]:
model.eval()
confident_mistakes = []  # list of (image, pred, true, confidence)

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        logits = model(X_batch)
        probs  = torch.softmax(logits, dim=1)
        preds  = logits.argmax(1)

        for i in range(len(y_batch)):
            pred = preds[i].item()
            true = y_batch[i].item()
            conf = probs[i, pred].item()

            if pred != true and conf > 0.70:
                confident_mistakes.append((X_batch[i], pred, true, conf))

print(f"Found {len(confident_mistakes)} confident mistakes (conf > 70%)")

# Show up to 8
show = confident_mistakes[:8]
fig, axes = plt.subplots(1, len(show), figsize=(14, 3))
if len(show) == 1: axes = [axes]

for ax, (img, pred, true, conf) in zip(axes, show):
    ax.imshow(img.squeeze().numpy(), cmap='gray')
    ax.set_title(f"Pred: {CLASS_NAMES[pred]}\n({conf:.0%})\nTrue: {CLASS_NAMES[true]}",
                 color='red', fontsize=7)
    ax.axis('off')

plt.suptitle("Confident Mistakes (>70% confidence, wrong class)", fontweight='bold')
plt.tight_layout()
plt.show()

# What do the confused images have in common?
# Most confident mistakes happen between visually similar classes:
# Shirt vs T-shirt/top vs Pullover (all tops, subtle shape differences)
# Coat vs Pullover (similar silhouette)
# Sneaker vs Ankle boot (both footwear with similar shapes)
# This makes intuitive sense — these are categories that even humans
# sometimes disagree on. The model is not making random errors;
# it's confusing things that genuinely look alike in 28x28 grayscale.